# Telco Churn – Feature Engineering & Baseline Models
**Branch:** `feature/churn-model`

Metric to optimise: **ROC-AUC** (F1 secondary). Test set stays locked – only the 80% train split is used here.

## 1. Load train split (same seed as EDA notebook)

In [1]:
import pandas as pd, numpy as np, os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

RANDOM_STATE = 42
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
try:
    df = pd.read_csv(URL)
except Exception:
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = df['Churn'].map({'No':0,'Yes':1})
X = df.drop(columns=['Churn','customerID']); y = df['Churn']

# identical split to EDA notebook; test part is discarded immediately
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
del X_test, y_test
print(X_train.shape, y_train.mean().round(3))

(5634, 19) 0.265


## 2. Feature engineering (5 new features)
These are stateless row-wise transforms (no statistics learned from data), so applying them before CV causes no leakage.

| Feature | Idea |
|---|---|
| `tenure_group` | Bucket tenure: 0-12, 13-24, 25-48, 49-72 months |
| `avg_monthly_spend` | TotalCharges / tenure (actual avg spend) |
| `charge_diff` | MonthlyCharges − avg_monthly_spend (recent price change signal) |
| `num_services` | Count of add-on services subscribed |
| `has_protection` | Has OnlineSecurity or TechSupport (stickiness) |
| `is_autopay` | Pays via automatic bank transfer / credit card |

In [2]:
SERVICE_COLS = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

def add_features(d):
    d = d.copy()
    d['tenure_group'] = pd.cut(d['tenure'], bins=[-1,12,24,48,72],
                               labels=['0-12','13-24','25-48','49-72']).astype(str)
    t = d['tenure'].replace(0, np.nan)
    d['avg_monthly_spend'] = (d['TotalCharges'] / t).fillna(d['MonthlyCharges'])
    d['charge_diff'] = d['MonthlyCharges'] - d['avg_monthly_spend']
    d['num_services'] = (d[SERVICE_COLS] == 'Yes').sum(axis=1)
    d['has_protection'] = ((d['OnlineSecurity']=='Yes') | (d['TechSupport']=='Yes')).astype(int)
    d['is_autopay'] = d['PaymentMethod'].str.contains('automatic', case=False).astype(int)
    return d

X_train_fe = add_features(X_train)
X_train_fe[['tenure_group','avg_monthly_spend','charge_diff','num_services','has_protection','is_autopay']].head()

,tenure_group,avg_monthly_spend,charge_diff,num_services,has_protection,is_autopay
3738,25-48,48.618571,0.581429,3,0,0
3151,13-24,76.770000,-1.670000,2,1,0
4860,13-24,45.411538,-4.861538,3,1,0
3867,25-48,73.296154,0.203846,5,0,1
3810,0-12,44.550000,0.000000,1,0,0


## 3. Preprocessing: ColumnTransformer

In [3]:
num_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_fe.select_dtypes(exclude=[np.number]).columns.tolist()
print('Numeric:', num_cols); print('Categorical:', cat_cols)

preprocess = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

Numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'charge_diff', 'num_services', 'has_protection', 'is_autopay']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']


## 4. Pipelines + 5-fold stratified CV (Dummy baseline included)
Imputer/scaler/encoder are fitted inside each fold only, so no leakage.

In [4]:
models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Dummy (stratified)':    DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':         RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    s = cross_validate(pipe, X_train_fe, y_train, cv=cv,
                       scoring={'roc_auc':'roc_auc','f1':'f1'}, n_jobs=-1)
    rows.append({'Model': name,
                 'CV ROC-AUC': round(s['test_roc_auc'].mean(), 3),
                 'ROC-AUC std': round(s['test_roc_auc'].std(), 3),
                 'CV F1': round(s['test_f1'].mean(), 3),
                 'F1 std': round(s['test_f1'].std(), 3)})

results = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
results.to_csv('results/baseline_results.csv', index=False)
results

,Model,CV ROC-AUC,ROC-AUC std,CV F1,F1 std
0,Dummy (most_frequent),0.500,0.000,0.000,0.000
1,Dummy (stratified),0.507,0.017,0.276,0.025
2,Logistic Regression,0.846,0.012,0.625,0.024
3,Random Forest,0.845,0.009,0.631,0.015


## 5. Observations
- Both real models should clearly beat the dummy baselines (ROC-AUC 0.5, F1 ~0 / ~0.27).
- Logistic Regression and Random Forest end up close; LR is simpler and more interpretable.
- Std across folds is small, so the scores are stable.
- Next step: hyperparameter tuning + threshold tuning, then final evaluation on the held-out test set.

(Fill in the actual numbers from your results table.)

Logistic Regression: ROC-AUC = 0.xx, F1 = 0.xx. Random Forest: ROC-AUC = 0.xx, F1 = 0.xx. Both models clearly beat the dummy baselines (ROC-AUC 0.50).

## 6. Commit
```bash
git add churn_baselines.ipynb results/baseline_results.csv
git commit -m "Add feature engineering, pipelines and baseline CV results"
git push origin feature/churn-model
```